In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/one-million-clicks-later/sample_submission.csv
/kaggle/input/one-million-clicks-later/train.csv
/kaggle/input/one-million-clicks-later/test.csv


In [18]:
import numpy as np


for df in [X, test_X]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(0, inplace=True)


for df in [X_train, X_valid]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(0, inplace=True)


/tmp/ipykernel_38/2160635319.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.replace([np.inf, -np.inf], np.nan, inplace=True)
/tmp/ipykernel_38/2160635319.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.fillna(0, inplace=True)


In [5]:
import pandas as pd


train = pd.read_csv('/kaggle/input/one-million-clicks-later/train.csv')
test = pd.read_csv('/kaggle/input/one-million-clicks-later/test.csv')


In [24]:
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Train columns:", train.columns.tolist())
print(train.info())


Train shape: (610310, 15)
Test shape: (152578, 14)
Train columns: ['user_id', 'video_id', 'video_duration', 'watch_time', 'liked', 'commented', 'subscribed_after', 'category', 'device', 'watch_time_of_day', 'recommended', 'clicked', 'timestamp', 'watch_percent', 'id']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 610310 entries, 0 to 610309
Data columns (total 15 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   user_id            610310 non-null  int64  
 1   video_id           610310 non-null  int64  
 2   video_duration     610310 non-null  float64
 3   watch_time         610310 non-null  float64
 4   liked              610310 non-null  int64  
 5   commented          610310 non-null  int64  
 6   subscribed_after   610310 non-null  int64  
 7   category           610310 non-null  int64  
 8   device             610310 non-null  int64  
 9   watch_time_of_day  610310 non-null  int64  
 10  recommended        610310 n

In [25]:
print("Clicked Value Counts:")
print(train['clicked'].value_counts())


Clicked Value Counts:
clicked
0.0    521090
1.0     89218
2.0         2
Name: count, dtype: int64


In [14]:
from sklearn.preprocessing import LabelEncoder


label_cols = ['category', 'device', 'liked', 'watch_time_of_day']


numeric_cols = ['video_id', 'video_duration', 'watch_time', 'recommended', 'watch_percent']


for col in label_cols:
    train[col] = train[col].fillna('missing').astype(str)
    test[col] = test[col].fillna('missing').astype(str)
    le = LabelEncoder()
    le.fit(list(train[col]) + list(test[col]))
    train[col] = le.transform(train[col])
    test[col] = le.transform(test[col])


for col in numeric_cols:
    train[col] = pd.to_numeric(train[col], errors='coerce').fillna(0)
    test[col] = pd.to_numeric(test[col], errors='coerce').fillna(0)


In [15]:
from sklearn.preprocessing import LabelEncoder


train = train.fillna('missing')
test = test.fillna('missing')


label_cols = ['category', 'device', 'liked', 'watch_time_of_day']

for col in label_cols:
    le = LabelEncoder()
    all_values = list(train[col].astype(str)) + list(test[col].astype(str))
    le.fit(all_values)
    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))


features = [
    'video_id', 'video_duration', 'watch_time', 'category',
    'device', 'watch_time_of_day', 'recommended', 'liked', 'watch_percent'
]



X = train[features]
y = train['clicked']
test_X = test[features]




In [16]:
from sklearn.model_selection import train_test_split
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)


In [19]:
from sklearn.model_selection import train_test_split
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.linear_model import LogisticRegression
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)
val_preds = logreg.predict(X_valid)
accuracy = (val_preds == y_valid).mean()
print("Validation Accuracy:", accuracy)


Validation Accuracy: 0.8542625878651833


In [21]:
val_preds = logreg.predict(X_valid)
accuracy = (val_preds == y_valid).mean()
print("Validation Accuracy:", accuracy)


Validation Accuracy: 0.8542625878651833


In [20]:
from sklearn.metrics import f1_score

f1 = f1_score(y_valid, val_preds)
print("Validation F1-Score:", f1)


Validation F1-Score: 0.0


In [22]:
from sklearn.metrics import f1_score

probs = logreg.predict_proba(X_valid)[:, 1]


for thresh in [0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5]:
    f1 = f1_score(y_valid, probs > thresh)
    print(f"F1-Score at threshold={thresh}: {f1}")


F1-Score at threshold=0.05: 0.25428209150888653
F1-Score at threshold=0.1: 0.2503661264624941
F1-Score at threshold=0.15: 0.22973358028209145
F1-Score at threshold=0.2: 0.1847168999364741
F1-Score at threshold=0.3: 0.04287723723141097
F1-Score at threshold=0.4: 0.00022455510020771345
F1-Score at threshold=0.5: 0.0


In [23]:

test_probs = logreg.predict_proba(test_X)[:, 1]

test_pred = (test_probs > 0.05).astype(int)


submission = pd.DataFrame({
    'id': test['id'],
    'clicked': test_pred
})

submission.to_csv('submission.csv', index=False)
print(submission.head())


       id  clicked
0   53363        1
1  293669        1
2   52195        1
3  260007        1
4  602213        1
